# Acquisition Functions

ALF ships the common acquisition functions in [`alf_tools`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/acquisition_functions/) — import and use them directly rather than reimplementing them. They all implement the same [`AcquisitionFunction`](https://instadeepai.github.io/alf/api/alf_core/optimizer/acquisition_function/) interface, so this tutorial exercises three of the built-ins and documents that interface — which is all you need to write your own when a built-in does not fit (see the Key Points at the end).

## 1. Imports

In [ ]:
import numpy as np
from alf_core.dataclasses import Candidate, LabelledCandidates, Modality, Predictions, State
from alf_core.dataset.base_dataset import BaseDataset, BaseDatasetConfig
from alf_core.model.base_model import BaseModel
from alf_core.surrogate.surrogate import Surrogate
from alf_core.utils.enums import ProblemType
from alf_tools.optimizer.acquisition_functions import UCB, CoreSet, UncertaintyBased

## 2. The Built-in Acquisition Functions

Each acquisition function implements `__call__(search_candidates, state)`: it receives a list of [`Candidate`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/candidate/) objects and the current [`State`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/state/) (holding the surrogate model and dataset), and returns [`LabelledCandidates`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/labelled_candidates/) whose labels are acquisition scores (higher = selected first). We use three of the built-ins below.

[`UncertaintyBased`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/acquisition_functions/uncertainty_based/) scores each candidate by its prediction variance — pure exploration. It requires an uncertainty-aware surrogate (e.g. an ensemble or a Gaussian process) whose `predict` populates `variances`, and raises if none are available.

[`UCB`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/acquisition_functions/ucb/) (Upper Confidence Bound) scores `mean + α·std`, trading exploitation against exploration via `alpha`: higher `alpha` weights uncertainty more (more exploration), lower `alpha` chases the highest predicted value (more exploitation).

[`CoreSet`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/acquisition_functions/core_set/) ignores the surrogate's predictions and instead greedily selects diverse candidates (greedy k-centres) that are far — in feature space — from the training set and from already-selected candidates. Scores are selection ranks (first picked = highest), so it favours under-explored regions.

## 3. Usage Example

To exercise these, we build a `State` with a toy surrogate (returns random means/variances) and a small training split for the diversity score to measure distance from.

In [ ]:
# Acquisition functions receive the full task `State`. We build one with:
#  - a surrogate that can predict (a toy model wrapped in `Surrogate`)
#  - a dataset whose training split the diversity score measures distance from
#    (see the "Extending Datasets" tutorial for the dataset pattern)
class InMemoryDataset(BaseDataset):
    """Minimal dataset backed by candidates and labels already in memory."""

    def __init__(self, candidates, labels, config):
        super().__init__(config)
        self._candidates = candidates
        self._labels = labels

    def load_dataset(self) -> LabelledCandidates:
        return LabelledCandidates(candidates=self._candidates, labels=self._labels)


class MockModel(BaseModel):
    """Toy surrogate that returns random means and variances."""

    def featurise(self, inputs):
        return np.array([c.data for c in inputs]).reshape(-1, 1)

    def train(self, train_data, val_data):
        pass

    def predict(self, candidate_points):
        n = len(candidate_points)
        return Predictions(means=np.random.rand(n), variances=np.random.rand(n) * 0.1)

    def sample(self, condition=None):
        return []

Assemble the dataset and `State`:

In [ ]:
# Some training data; the diversity score measures distance from the training split
train_xs = np.linspace(0.0, 9.0, 10)
training_candidates = [Candidate(data=float(x), modality=Modality.TABULAR) for x in train_xs]
training_labels = np.array([float(x) for x in train_xs])
config = BaseDatasetConfig(
    name="demo",
    modality=Modality.TABULAR,
    seed=0,
    train_ratio=0.6,
    validation_frac=0.2,
    test_ratio=0.2,
    split_type="random",
    problem_type=ProblemType.REGRESSION,
)
dataset = InMemoryDataset(training_candidates, training_labels, config)
dataset.setup()

state = State(
    dataset=dataset,
    surrogate=Surrogate(model=MockModel()),
    round=1,
    acq_batch_size=5,
)

Generate a pool of search candidates and score them with each strategy, taking the top 5 via `get_top_k`:

In [ ]:
# Generate search candidates
search_candidates = [Candidate(data=x, modality=Modality.TABULAR) for x in np.linspace(0, 10, 20)]

# Apply different built-in acquisition functions
print("=== Uncertainty-Based ===")
uncertainty_acq = UncertaintyBased()
scored = uncertainty_acq(search_candidates, state)
top_5 = scored.get_top_k(5)
print(f"Top 5 scores: {top_5.labels}")
print(f"Top 5 candidates: {[c.data for c in top_5.candidates]}")

print("\n=== Upper Confidence Bound ===")
ucb_acq = UCB(alpha=2.0)
scored = ucb_acq(search_candidates, state)
top_5 = scored.get_top_k(5)
print(f"Top 5 scores: {top_5.labels}")

print("\n=== Core-Set (diversity) ===")
coreset_acq = CoreSet()
scored = coreset_acq(search_candidates, state)
top_5 = scored.get_top_k(5)
print(f"Top 5 scores: {top_5.labels}")
print(f"Top 5 candidates: {[c.data for c in top_5.candidates]}")

## Key Points

- **Required method**: `__call__(search_candidates, state)` must return `LabelledCandidates`
- **Input**: Receives unlabelled candidates from search function and current task state
- **Output**: Return candidates with acquisition scores as labels (higher = better)
- **Surrogate access**: Use `state.surrogate.predict()` to get predictions
- **Training data**: Access via `state.dataset` to avoid re-selecting points
- **Scoring**: Higher scores indicate candidates worth evaluating
- **Selection**: Framework automatically selects top-k candidates based on scores

Common acquisition strategies:
- **Greedy**: Select highest predicted value (exploitation)
- **Uncertainty**: Select most uncertain predictions (pure exploration)
- **UCB**: Balance exploitation and exploration with beta parameter
- **Expected Improvement**: Probability of improvement over current best
- **Thompson Sampling**: Sample from posterior distribution
- **Diversity**: Maximize distance from existing points

See existing implementations in `tools/alf_tools/optimizer/acquisition_functions/` for more examples.